# 02 — Wall graph (Hough + optional RF patches)

Compare detected wall length on the **synthetic** image to ground truth (approximate; Hough is sensitive to parameters).

In [ ]:
from pathlib import Path
import sys

ROOT = Path().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from blueprint_estimator.synthetic import synthetic_ingest, total_gt_length_px
from blueprint_estimator.wall_graph_cv import image_to_wall_segments, total_segment_length
from blueprint_estimator.wall_ml import (
    build_synthetic_patch_dataset,
    train_wall_patch_rf,
    score_segments_for_wall,
)

syn = synthetic_ingest()
img = syn.image_bgr
segs, graph = image_to_wall_segments(img, merge=True)
print("detected segments:", len(segs), "graph edges:", len(graph.edges))
print("detected total px (sum segments):", total_segment_length(segs))
print("GT total px:", total_gt_length_px(syn.segments_draw))

X, y = build_synthetic_patch_dataset(img, syn.segments_draw)
clf = train_wall_patch_rf(X, y)
scores = score_segments_for_wall(clf, img, segs)
print("mean wall proba (detected):", sum(scores) / max(len(scores), 1))